# Лаб 2. Random Forest

In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier as SklearnRF
from sklearn.metrics import accuracy_score
from sklearn.model_selection import ParameterGrid
from sklearn.datasets import fetch_openml

from source.model import RandomForestClassifier

## Загрузка данных

Heart Disease (Statlog) — тот же датасет что в лаб 1

In [2]:
data = fetch_openml('heart-statlog', version=1, as_frame=True)
df = data.frame
print(df.shape)
df.head()

(270, 14)


,age,sex,chest,resting_blood_pressure,serum_cholestoral,fasting_blood_sugar,resting_electrocardiographic_results,maximum_heart_rate_achieved,exercise_induced_angina,oldpeak,slope,number_of_major_vessels,thal,class
0,70,1,4,130,322,0,2,109,0,2.4,2,3,3,present
1,67,0,3,115,564,0,2,160,0,1.6,2,0,7,absent
2,57,1,2,124,261,0,0,141,0,0.3,1,0,7,present
3,64,1,4,128,263,0,0,105,1,0.2,2,1,7,absent
4,74,0,2,120,269,0,2,121,1,0.2,1,1,3,absent


In [3]:
target_col = df.columns[-1]
feature_names = [c for c in df.columns if c != target_col]

X = df[feature_names].values.astype(float)
y_raw = df[target_col].values
classes_map = {v: i for i, v in enumerate(np.unique(y_raw))}
y = np.array([classes_map[v] for v in y_raw])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'train={len(y_train)} test={len(y_test)}')
print(f'признаков: {len(feature_names)}')
print(f'классы: {classes_map}')

train=216 test=54
признаков: 13
классы: {'absent': 0, 'present': 1}


## Grid Search по OOB

In [4]:
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_features': ['sqrt', 'log2', 0.5],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5],
}

best_oob = -1
best_params = None

for params in ParameterGrid(param_grid):
    rf = RandomForestClassifier(random_state=42, **params)
    rf.fit(X_train, y_train)
    if rf.oob_score_ > best_oob:
        best_oob = rf.oob_score_
        best_params = params

print(f'лучший OOB: {best_oob:.4f}')
print(f'параметры: {best_params}')

лучший OOB: 0.8519
параметры: {'max_depth': 15, 'max_features': 0.5, 'min_samples_split': 5, 'n_estimators': 150}


## Обучение с лучшими параметрами

In [5]:
t0 = time.time()
my_rf = RandomForestClassifier(random_state=42, **best_params)
my_rf.fit(X_train, y_train)
my_time = time.time() - t0

my_pred = my_rf.predict(X_test)
my_acc = accuracy_score(y_test, my_pred)
print(f'accuracy={my_acc:.4f} oob={my_rf.oob_score_:.4f} time={my_time:.3f}s')

accuracy=0.8333 oob=0.8519 time=0.075s


## Важность признаков (OOB^j)

In [6]:
importances = my_rf.get_feature_importances_oob(X_train, y_train, random_state=42)

imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print(imp_df.to_string(index=False))

                             feature  importance
                               chest    0.270833
                                thal    0.208333
             number_of_major_vessels    0.166667
                   serum_cholestoral    0.104167
                             oldpeak    0.104167
                                 age    0.083333
                                 sex    0.020833
              resting_blood_pressure    0.020833
             exercise_induced_angina    0.020833
                               slope    0.020833
                 fasting_blood_sugar    0.000000
         maximum_heart_rate_achieved    0.000000
resting_electrocardiographic_results   -0.020833


## Сравнение со sklearn

In [7]:
t0 = time.time()
sk_rf = SklearnRF(
    oob_score=True,
    random_state=42,
    **best_params
)
sk_rf.fit(X_train, y_train)
sk_time = time.time() - t0

sk_pred = sk_rf.predict(X_test)
sk_acc = accuracy_score(y_test, sk_pred)
print(f'sklearn accuracy={sk_acc:.4f} oob={sk_rf.oob_score_:.4f} time={sk_time:.3f}s')

sklearn accuracy=0.8148 oob=0.8287 time=0.112s


## Итоговая таблица

In [8]:
pd.DataFrame({
    'модель': ['Custom RF', 'sklearn RF'],
    'accuracy': [my_acc, sk_acc],
    'OOB score': [my_rf.oob_score_, sk_rf.oob_score_],
    'время (с)': [my_time, sk_time],
})

,модель,accuracy,OOB score,время (с)
0,Custom RF,0.833333,0.851852,0.075494
1,sklearn RF,0.814815,0.828704,0.112302
